# SimpleMD — In-Notebook Visualization

This notebook runs a short Lennard-Jones simulation and visualizes the particle trajectory using [py3Dmol](https://github.com/3dmol/3Dmol.js), which renders interactively in VS Code, Jupyter, and Google Colab.

For more advanced visualization (depth cueing, rendering), see the VMD instructions in the SimpleMD manual (Ch. 5).

**Controls**
- Drag to rotate
- Scroll to zoom
- Right-click drag to pan

In [ ]:
# --- Google Colab only: clone the repo and install py3Dmol ---
# Skip this cell if running locally.
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/emfurst/SimpleMD.git
    %cd SimpleMD
    !pip install py3Dmol --quiet

In [1]:
import os
import py3Dmol
import CHEG231MD as md

## Run the simulation

Set up and equilibrate, then run a short production trajectory.
Adjust `nn`, `rho`, and `vmax` to explore different conditions.

In [39]:
nn   = 5      # particles per side (N = nn^3)
rho  = 0.4    # reduced number density
vmax = 1.5    # initial max velocity

sim = md.MDSimulation(nn, rho, vmax)

# Equilibration
for _ in range(200):
    sim.move()
print(f"Equilibrated:  T* = {sim.T:.3f}  P* = {sim.P:.3f}")

Equilibrated:  T* = 0.614  P* = -0.392


In [40]:
XYZ_FILE = "MDvis.xyz"

def write_frame(f, sim, timestep):
    f.write(f"{sim.N}\n")
    f.write(f"Timestep: {timestep}\n")
    for row in sim.x:
        f.write(f"Ar {row[0]:.6f} {row[1]:.6f} {row[2]:.6f}\n")

# Production run — write every 5th frame
with open(XYZ_FILE, "w") as f:
    for t in range(1, 501):
        sim.move()
        if t % 10 == 0:
            write_frame(f, sim, t)

print(f"Wrote trajectory to {XYZ_FILE}  (T* = {sim.T:.3f}, P* = {sim.P:.3f})")

Wrote trajectory to MDvis.xyz  (T* = 0.836, P* = -0.122)


## Animated trajectory

The viewer below plays through all saved frames.
The wireframe box shows the periodic simulation cell.

In [50]:
with open(XYZ_FILE) as f:
    xyz = f.read()

view = py3Dmol.view(width=600, height=450)
view.addModelsAsFrames(xyz, "xyz")
view.setStyle({"sphere": {"radius": 0.5, "color": "red", "opacity": 1}})
view.setBackgroundColor("white")

# Draw the periodic simulation box
c = sim.L / 2
view.addBox({"center": {"x": c, "y": c, "z": c},
             "dimensions": {"w": sim.L, "h": sim.L, "d": sim.L},
             "color": "white", "opacity": 0.5, "wireframe": False})

view.zoomTo()
view.animate({"loop": "forward", "interval": 100})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.